# ChakraTransformer Kaggle Standalone Evaluation
**Combo #6 (ViT-Large 384)** — 100% standalone, no `src/` folder required.

Evaluates Kvasir-SEG (held-out 15% test split), CVC-ClinicDB (zero-shot), ETIS-Larib (zero-shot).

### Required Dataset Mounts
- `/kaggle/input/datasets/gokulrocky/kaggle-upload-zip4` (weights)
- `/kaggle/input/notebooks/ivannikov2002/kvasir-seg-data-polyp-segmentation-detection`
- `/kaggle/input/notebooks/ahaan2/cvc-clinicdb`
- `/kaggle/input/notebooks/tamimm91437/etis-laribpolypdb`

In [ ]:
# CELL 1: ENVIRONMENT SETUP
# NOTE: torchvision is intentionally NOT installed via pip.
# Kaggle base image has torch + torchvision pinned to matching CUDA versions.
# Installing torchvision via pip would override the precompiled CUDA operators -> crash.
import subprocess
subprocess.run(['pip', 'install', '-q', 'timm', 'opencv-python-headless', 'albumentations'], check=True)

import os, cv2, torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB  | AMP FP16 Enabled')
else:
    print('WARNING: CUDA unavailable, running on CPU.')
print(f'Using device: {device}')

In [ ]:
# CELL 2: EXACT CHAKRATRANSFORMER ARCHITECTURE
# ---------------------------------------------------------------------------
# This is a 1-to-1 copy of src/chakra_transformer/transformer_segmenter.py.
# The layer names, dtypes, and tensor shapes MUST match the saved checkpoint
# so that model.load_state_dict(strict=True) succeeds without any key errors.
# ---------------------------------------------------------------------------
import timm

class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, backbone_name='vit_large_patch16_384', pretrained=False, num_classes=1):
        super(ChakraTransformerSegmenter, self).__init__()

        # timm ViT backbone (features_only=False to use forward_features)
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=False)

        # ViT-Large embed_dim = 1024
        self.embed_dim = self.backbone.embed_dim

        # Decode head — EXACT sequential matching the checkpoint keys:
        # decode_head.0.weight / decode_head.1.weight / ... / decode_head.6.weight
        # Patch grid at 384px input = 384/16 = 24x24
        # ConvTranspose2d(k=4,s=4): 24 -> 96  then  ConvTranspose2d(k=4,s=4): 96 -> 384
        self.decode_head = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 256, kernel_size=4, stride=4),  # idx 0
            nn.BatchNorm2d(256),                                                # idx 1
            nn.ReLU(inplace=True),                                              # idx 2
            nn.ConvTranspose2d(256, 64,  kernel_size=4, stride=4),             # idx 3
            nn.BatchNorm2d(64),                                                 # idx 4
            nn.ReLU(inplace=True),                                              # idx 5
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1)               # idx 6
        )
        # MC Dropout layers — keys: dropout1.* and dropout2.*
        self.dropout1 = nn.Dropout2d(p=0.5)
        self.dropout2 = nn.Dropout2d(p=0.5)

    def forward(self, x):
        B, C, H, W = x.shape
        features = self.backbone.forward_features(x)

        if features.dim() == 3:
            # Safe CLS token strip — count actual expected patches.
            # Do NOT use self.backbone.global_pool attribute (fragile across timm versions).
            expected_patches = (H // 16) * (W // 16)
            if features.shape[1] == expected_patches + 1:
                features = features[:, 1:, :]  # drop CLS token
            features = features.transpose(1, 2).contiguous().view(B, self.embed_dim, H // 16, W // 16)

        # Decode with MC Dropout injected at the correct layer indices
        x_dec = features
        for i, layer in enumerate(self.decode_head):
            x_dec = layer(x_dec)
            if i == 2:  x_dec = self.dropout1(x_dec)
            elif i == 5: x_dec = self.dropout2(x_dec)
        logits = x_dec

        # Bilinear upsample if output resolution drifts from input
        if logits.shape[2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        return logits

print('ChakraTransformerSegmenter architecture defined (exact match to checkpoint).')

In [ ]:
# CELL 3: EVALUATION DATASET AND METRICS
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

class EvalPolypDataset(Dataset):
    # Augmentation-free evaluation dataset.
    # Accepts a pre-sorted image file list and a stem->mask_path dict.
    def __init__(self, file_paths, mask_dict, img_size=384):
        self.file_paths = file_paths
        self.mask_dict  = mask_dict
        self.img_size   = img_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):  return len(self.file_paths)

    def __getitem__(self, idx):
        img_path  = self.file_paths[idx]
        mask_path = self.mask_dict.get(img_path.stem)

        img = cv2.imread(str(img_path))
        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)

        if mask_path is not None and Path(mask_path).exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
            else:
                mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        else:
            mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)

        img_t  = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0
        img_t  = (img_t - self.mean) / self.std
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


def compute_metrics(model, loader, device):
    model.eval().to(device)
    dices, ious, precs, recs = [], [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            gts   = masks.numpy()
            for i in range(len(probs)):
                p  = (probs[i, 0] > 0.5).astype(np.float32)
                g  = (gts[i,   0] > 0.5).astype(np.float32)
                tp = (p * g).sum()
                fp = (p * (1.0 - g)).sum()
                fn = ((1.0 - p) * g).sum()
                dices.append((2.0*tp + 1e-6) / (2.0*tp + fp + fn + 1e-6))
                ious.append(  (tp    + 1e-6) / (tp + fp + fn + 1e-6))
                precs.append( (tp    + 1e-6) / (tp + fp + 1e-6))
                recs.append(  (tp    + 1e-6) / (tp + fn + 1e-6))
    return {'dice': float(np.mean(dices)), 'iou': float(np.mean(ious)),
            'precision': float(np.mean(precs)), 'recall': float(np.mean(recs))}

print('EvalPolypDataset and compute_metrics defined.')

In [ ]:
# CELL 4: SAFE MODEL WEIGHT LOADING
print('Initializing ChakraTransformer (ViT-Large 384, pretrained=False)...')
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)

WEIGHT_FILENAME = 'chakra_transformer_best.pth'
candidate_paths = [
    Path('/kaggle/working') / WEIGHT_FILENAME,
    Path('/kaggle/working/weights') / WEIGHT_FILENAME,
    Path('.') / WEIGHT_FILENAME,
]
for p in Path('/kaggle/input').rglob(WEIGHT_FILENAME):
    candidate_paths.append(p)

weight_path = next((p for p in candidate_paths if p.exists()), None)

if weight_path is None:
    raise FileNotFoundError(
        '\n' + '='*62 + '\n'
        f' ERROR: {WEIGHT_FILENAME!r} not found anywhere!\n'
        + '='*62 + '\n'
        ' Run Combo6_ChakraTransformer.ipynb first to train the model,\n'
        ' then mount the /kaggle/working output as a dataset here.\n'
        + '='*62
    )

print(f'Loading weights from: {weight_path}')
state_dict = torch.load(weight_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model ready. Parameters: {n_params/1e6:.2f}M')

In [ ]:
# CELL 5: ROBUST DATASET DISCOVERY + EVALUATION DRIVER

def discover_img_mask_paths(root_dir, img_dir_names, mask_dir_names):
    root = Path(root_dir)
    if not root.exists(): return [], {}

    img_lo  = {n.lower() for n in img_dir_names}
    mask_lo = {n.lower() for n in mask_dir_names}

    img_dirs, mask_dirs = [], []
    for p in root.rglob('*'):
        if not p.is_dir(): continue
        lo = p.name.lower()
        if lo in img_lo:  img_dirs.append(p)
        if lo in mask_lo: mask_dirs.append(p)

    # Sort by filename — CRITICAL: must match training notebook sort order to prevent leakage
    img_files = []
    for d in img_dirs:
        img_files += sorted([f for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTS], key=lambda x: x.name)

    mask_dict = {}
    for d in mask_dirs:
        for f in d.iterdir():
            if f.suffix.lower() in IMAGE_EXTS:
                mask_dict[f.stem] = f

    return img_files, mask_dict


def run_evaluation(dataset_root, dataset_name, img_dir_names, mask_dir_names, is_kvasir=False):
    SEP = '=' * 60
    print(f'\n{SEP}\n  {dataset_name}\n{SEP}')

    img_files, mask_dict = discover_img_mask_paths(dataset_root, img_dir_names, mask_dir_names)
    if not img_files:
        print(f'  WARNING: No images found under {dataset_root}. Skipping.')
        return None
    if not mask_dict:
        print(f'  WARNING: No masks found under {dataset_root}. Skipping.')
        return None

    print(f'  Discovered {len(img_files)} images, {len(mask_dict)} masks')

    if is_kvasir:
        # EXACT replication of Combo6 tri-split (70/15/15, seed=42)
        # img_files is sorted by name — identical to training notebook's sorted(Path.glob('*'))
        np.random.seed(42)
        perm    = np.random.permutation(len(img_files))
        n_train = int(0.70 * len(img_files))
        n_cal   = int(0.15 * len(img_files))
        idx_test = perm[n_train + n_cal:]
        eval_files = [img_files[i] for i in idx_test]
        print(f'  Held-out test split (70/15/15, seed=42): {len(eval_files)} images')
    else:
        eval_files = img_files
        print(f'  Zero-shot: {len(eval_files)} images')

    dataset = EvalPolypDataset(eval_files, mask_dict, img_size=384)
    loader  = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
    metrics = compute_metrics(model, loader, device)

    print(f"\n  Dice (DSC) : {metrics['dice']:.4f}")
    print(f"  mIoU       : {metrics['iou']:.4f}")
    print(f"  Precision  : {metrics['precision']:.4f}")
    print(f"  Recall     : {metrics['recall']:.4f}")
    return metrics

print('Evaluation driver defined.')

## 1. Kvasir-SEG (Held-Out 15% Test Split)

In [ ]:
kvasir_metrics = run_evaluation(
    '/kaggle/input/notebooks/ivannikov2002/kvasir-seg-data-polyp-segmentation-detection',
    'Kvasir-SEG',
    img_dir_names=['images', 'Images'],
    mask_dir_names=['masks', 'Masks'],
    is_kvasir=True
)
# Fallback: try user's primary zip upload
if kvasir_metrics is None:
    kvasir_metrics = run_evaluation(
        '/kaggle/input/datasets/gokulrocky/kaggle-upload-zip4',
        'Kvasir-SEG (from zip upload)',
        img_dir_names=['images', 'Images'],
        mask_dir_names=['masks', 'Masks'],
        is_kvasir=True
    )

## 2. CVC-ClinicDB (Zero-Shot)

In [ ]:
cvc_metrics = run_evaluation(
    '/kaggle/input/notebooks/ahaan2/cvc-clinicdb',
    'CVC-ClinicDB',
    img_dir_names=['images', 'Images', 'original', 'Original'],
    mask_dir_names=['masks', 'Masks', 'ground truth', 'Ground Truth', 'groundtruth'],
    is_kvasir=False
)

## 3. ETIS-Larib (Zero-Shot)

In [ ]:
etis_metrics = run_evaluation(
    '/kaggle/input/notebooks/tamimm91437/etis-laribpolypdb',
    'ETIS-Larib',
    img_dir_names=['images', 'Images', 'etis-laribpolypdb'],
    mask_dir_names=['masks', 'Masks', 'etis-laribpolypdb-gt'],
    is_kvasir=False
)

## Final Summary Table

In [ ]:
results = {
    'Kvasir-SEG':   kvasir_metrics,
    'CVC-ClinicDB': cvc_metrics,
    'ETIS-Larib':   etis_metrics,
}

header = f"{'Dataset':<22} | {'Dice (DSC)':<12} | {'mIoU':<10} | {'Precision':<12} | {'Recall':<10}"
sep    = '=' * len(header)
print(f'\n{sep}')
print('  CHAKRATRANSFORMER (ViT-LARGE 384) — FINAL EVALUATION SUMMARY')
print(sep)
print(header)
print('-' * len(header))
for name, m in results.items():
    if m is None:
        print(f'{name:<22} | (not evaluated)')
    else:
        print(f"{name:<22} | {m['dice']:<12.4f} | {m['iou']:<10.4f} | {m['precision']:<12.4f} | {m['recall']:<10.4f}")
print(sep)